In [ ]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
BASE_PATH = os.environ.get("BASE_PATH")
calendar = pd.read_parquet(f'{BASE_PATH}/data/processed/calendar_cleaned.parquet')
sales = pd.read_parquet(f'{BASE_PATH}/data/processed/sales_train_cleaned.parquet')
price = pd.read_parquet(f'{BASE_PATH}/data/processed/sell_prices_cleaning.parquet')

In [6]:
sales_long = pd.melt(
    sales,
    id_vars=["item_store_id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
    var_name="day_number",
    value_name="sales"
)
sales_long["day_number"] = sales_long["day_number"].astype("int16")

In [4]:
calendar.info()

<class 'pandas.DataFrame'>
RangeIndex: 1969 entries, 0 to 1968
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               1969 non-null   datetime64[us]
 1   walmart_year_week  1969 non-null   int32         
 2   weekday            1969 non-null   category      
 3   day_number         1969 non-null   int16         
 4   event_name         1969 non-null   category      
 5   event_type         1969 non-null   category      
 6   snap_CA            1969 non-null   bool          
 7   snap_TX            1969 non-null   bool          
 8   snap_WI            1969 non-null   bool          
 9   month_name         1969 non-null   category      
dtypes: bool(3), category(4), datetime64[us](1), int16(1), int32(1)
memory usage: 41.4 KB


In [7]:
sales_long.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 59181090 entries, 0 to 59181089
Data columns (total 8 columns):
 #   Column         Dtype   
---  ------         -----   
 0   item_store_id  str     
 1   item_id        category
 2   dept_id        category
 3   cat_id         category
 4   store_id       category
 5   state_id       category
 6   day_number     int16   
 7   sales          Int16   
dtypes: Int16(1), category(5), int16(1), str(1)
memory usage: 2.6 GB


In [8]:
# the merge will be depending on day_number
sales_calendar = sales_long.merge(
    calendar, 
    on='day_number',
    how='left'
)

In [10]:
print(sales_calendar.shape)
print(sales_calendar.isna().sum())

(59181090, 17)
item_store_id        0
item_id              0
dept_id              0
cat_id               0
store_id             0
state_id             0
day_number           0
sales                0
date                 0
walmart_year_week    0
weekday              0
event_name           0
event_type           0
snap_CA              0
snap_TX              0
snap_WI              0
month_name           0
dtype: int64


In [ ]:
df_merged = sales_calendar.merge(
    price,
    on=["store_id", "item_id", "walmart_year_week"],
    how="left"
)

In [12]:
print(df_merged.shape)
print(df_merged["sell_price"].isnull().sum())

(59181090, 18)
12299413


In [ ]:
df_merged.to_parquet(f'{BASE_PATH}/data/processed/df_merged.parquet', index=False)
